In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_C.xpt"
csv_file = DATA_DIR / "HIQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 HIQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED VARIABLES
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HID010",
    "HID030A",
    "HID030B",
    "HID030C",
    "HID030D",
    "HID030E",
    "HID040",
    "HIQ210",
    "HIQ220"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match HIQ_C.")
else:
    raise ValueError(
        "HIQ_C column structure differs from expected."
    )


# ============================================================
# 4. CHECK KNOWN PANDAS/XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{total_zero:,}"
)

print(
    "Tiny artifact values:",
    f"{total_tiny:,}"
)


if total_tiny > 0:

    print("\nVariables containing tiny values:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    # HIQ_C does NOT need zero restoration.
    # Stop instead of changing anything.

    raise ValueError(
        "STOP: Unexpected tiny-value artifact found in HIQ_C. "
        "Do NOT automatically convert it to zero."
    )


print(
    "PASS: No tiny zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# Fidelity rule:
#
# Never round a genuine decimal.
#
# First verify every numeric value is whole-number.
# If a decimal unexpectedly exists, STOP.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine fractional values found in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "These columns will NOT be converted:\n"
        + str(fractional_columns)
    )


print(
    "PASS: All HIQ_C values are whole-number "
    "codes/identifiers."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN CHECK ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE CODE VARIABLES
# ============================================================

CODE_COLUMNS = [
    "HID010",
    "HID030A",
    "HID030B",
    "HID030C",
    "HID030D",
    "HID030E",
    "HID040",
    "HIQ210",
    "HIQ220"
]


#
# We already verified there are no real decimal values.
#
# Therefore:
#
# 1.0 -> 1
# 2.0 -> 2
# 9.0 -> 9
#
# Missing stays missing.
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: HIQ questionnaire codes formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 10122:
    print(
        "PASS: Row count = 10,122"
    )
else:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) == 10:
    print(
        "PASS: Column count = 10"
    )
else:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


# ============================================================
# 9. CDC FREQUENCY DEFINITIONS
# ============================================================

EXPECTED = {

    "HID010": {
        "counts": {
            1: 8263,
            2: 1708,
            7: 3,
            9: 15
        },
        "missing": 133
    },

    "HID030A": {
        "counts": {
            1: 4790,
            2: 3435,
            7: 3,
            9: 35
        },
        "missing": 1859
    },

    "HID030B": {
        "counts": {
            1: 1483,
            2: 6742,
            7: 3,
            9: 35
        },
        "missing": 1859
    },

    "HID030C": {
        "counts": {
            1: 2101,
            2: 6124,
            7: 3,
            9: 35
        },
        "missing": 1859
    },

    "HID030D": {
        "counts": {
            1: 821,
            2: 7404,
            7: 3,
            9: 35
        },
        "missing": 1859
    },

    "HID030E": {
        "counts": {
            1: 216,
            2: 8009,
            7: 3,
            9: 35
        },
        "missing": 1859
    },

    "HID040": {
        "counts": {
            1: 5805,
            2: 2150,
            7: 2,
            9: 264
        },
        "missing": 1901
    },

    "HIQ210": {
        "counts": {
            1: 575,
            2: 7637,
            7: 1,
            9: 12
        },
        "missing": 1897
    },

    "HIQ220": {
        "counts": {
            1: 319,
            2: 178,
            3: 289,
            4: 386,
            5: 497,
            7: 2,
            9: 43
        },
        "missing": 8408
    }
}


# ============================================================
# 10. RUN CDC FREQUENCY CHECK
# ============================================================

print("\n--- CDC FREQUENCY CHECK ---")


frequency_failures = []


for col, spec in EXPECTED.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    counts_match = (
        actual_counts
        ==
        spec["counts"]
    )

    missing_match = (
        actual_missing
        ==
        spec["missing"]
    )


    if counts_match and missing_match:

        print(
            f"PASS: {col}"
        )

    else:

        frequency_failures.append(
            col
        )

        print(
            f"\nFAIL: {col}"
        )

        print(
            "Expected counts:",
            spec["counts"]
        )

        print(
            "Actual counts:",
            actual_counts
        )

        print(
            "Expected missing:",
            spec["missing"]
        )

        print(
            "Actual missing:",
            actual_missing
        )


if frequency_failures:

    raise ValueError(
        "CDC frequency mismatch in:\n"
        + str(frequency_failures)
    )


print(
    "\nPASS: All HIQ_C frequencies "
    "match CDC."
)


# ============================================================
# 11. VERIFY VALUES BEFORE EXPORT
# ============================================================

#
# There is NO zero correction in HIQ_C.
#
# Therefore numeric values must match the original
# XPT exactly.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 12. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 13. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 14. EXACT XPT -> CSV VALUE FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        value_failures.append(
            col
        )

        print(
            f"FAIL: {col} differs from XPT."
        )


if value_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(value_failures)
    )


print(
    "\nPASS: Every HIQ_C numeric value "
    "matches the source XPT."
)


# ============================================================
# 15. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )

else:

    raise ValueError(
        "FAIL: Missing-value positions changed."
    )


# ============================================================
# 16. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


# ============================================================
# 17. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific-notation occurrences:",
    scientific_count
)


if scientific_count == 0:

    print(
        "PASS: No scientific notation."
    )


# ============================================================
# 18. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    csv_df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Tiny artifact values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact exists."
    )

else:

    raise ValueError(
        "Tiny-value artifact exists in CSV."
    )


print("\n" + "=" * 75)
print("HIQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 HIQ_C XPT -> CSV
Rows: 10,122
Columns: 10

Variables:
['SEQN', 'HID010', 'HID030A', 'HID030B', 'HID030C', 'HID030D', 'HID030E', 'HID040', 'HIQ210', 'HIQ220']

PASS: Column names/order match HIQ_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: All HIQ_C values are whole-number codes/identifiers.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: HIQ questionnaire codes formatted without unnecessary .0.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 10,122
PASS: Column count = 10

--- CDC FREQUENCY CHECK ---
PASS: HID010
PASS: HID030A
PASS: HID030B
PASS: HID030C
PASS: HID030D
PASS: HID030E
PASS: HID040
PASS: HIQ210
PASS: HIQ220

PASS: All HIQ_C frequencies match CDC.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: SEQN       preserved.
PASS: HID010     preserved.
PASS: HID030A    preserved.
PASS: HID030B    preserved.
PASS: HID030